In [1]:
import os
from dotenv import load_dotenv
import tushare as ts

# Walks up from current directory to find .env
load_dotenv()

token = os.getenv("TUSHARE_TOKEN")

# Defensive check — fail loudly if the token isn't found
if not token:
    raise RuntimeError(
        "TUSHARE_TOKEN not found. Check that .env exists at project root "
        "and contains TUSHARE_TOKEN=your_token"
    )

pro = ts.pro_api(token)

# Sanity check the connection — this should print a small DataFrame
df = pro.stock_basic(list_status='L', fields='ts_code,name', limit=5)
print(df)

     ts_code   name
0  000001.SZ   平安银行
1  000002.SZ    万科Ａ
2  000004.SZ  *ST国华
3  000006.SZ   深振业Ａ
4  000007.SZ    全新好


In [2]:
# diagnose_mcap_disagreement.py
import pandas as pd
from pathlib import Path

# Reload both sides
new_df = pd.read_csv(
    "data/candidates/candidates_2024-12-31.csv",
    dtype={"ts_code": str}
)

s1 = pd.read_csv(
    "data/kdata_2024-12-31.csv",
    dtype={"code": str, "tradestatus": str, "isST": str}
)
for c in ("close", "volume", "amount", "turn"):
    s1[c] = pd.to_numeric(s1[c], errors="coerce")
s1 = s1[s1["tradestatus"] == "1"]
s1 = s1[s1["isST"] == "0"]
s1 = s1[(s1["volume"] > 0) & (s1["turn"] > 0)].copy()

# Re-derive baostock float_mcap explicitly so we can inspect components
s1["bs_implied_float_shares"] = s1["volume"] / (s1["turn"] / 100)
s1["bs_float_mcap"] = s1["close"] * s1["bs_implied_float_shares"]
s1["bs_circ_mv_yi"] = s1["bs_float_mcap"] / 1e8

# Convert codes
s1["ts_code"] = s1["code"].apply(
    lambda c: f"{c.split('.')[1]}.{c.split('.')[0].upper()}"
)

# Merge with all components visible
keep_new = ["ts_code", "close", "vol", "turnover_rate",
            "total_share", "float_share", "total_mv", "circ_mv", "circ_mv_yi"]
keep_s1 = ["ts_code", "close", "volume", "turn",
           "bs_implied_float_shares", "bs_circ_mv_yi"]

merged = new_df[keep_new].merge(
    s1[keep_s1], on="ts_code", suffixes=("_ts", "_bs")
)

merged["diff_pct"] = (
    (merged["circ_mv_yi"] - merged["bs_circ_mv_yi"]).abs()
    / merged["bs_circ_mv_yi"] * 100
)

# H1 check: do float_share counts disagree?
# Tushare's float_share is in 万股. baostock's implied is in raw shares.
# Convert baostock to 万股 for direct comparison.
merged["bs_implied_float_wan"] = merged["bs_implied_float_shares"] / 10_000
merged["float_share_diff_pct"] = (
    (merged["float_share"] - merged["bs_implied_float_wan"]).abs()
    / merged["float_share"] * 100
)

# H2 check: how does disagreement scale with turnover?
print("=== Disagreement vs turnover ===")
turn_buckets = pd.cut(
    merged["turnover_rate"],
    bins=[0, 0.1, 0.5, 1, 2, 5, 100],
    labels=["<0.1%", "0.1-0.5%", "0.5-1%", "1-2%", "2-5%", ">5%"]
)
print(merged.groupby(turn_buckets, observed=True)["diff_pct"].agg(
    ["count", "median", "max"]
).round(2))

# Component-level inspection of worst offenders
print("\n=== Top 15 worst disagreements ===")
worst = merged.nlargest(15, "diff_pct")[[
    "ts_code", "close_ts", "close_bs", "turn", "turnover_rate",
    "float_share", "bs_implied_float_wan", "float_share_diff_pct",
    "circ_mv_yi", "bs_circ_mv_yi", "diff_pct"
]]
print(worst.to_string(index=False))

# H1 check at the population level
print("\n=== Float-share disagreement distribution ===")
print(f"  median: {merged['float_share_diff_pct'].median():.2f}%")
print(f"  P95:    {merged['float_share_diff_pct'].quantile(0.95):.2f}%")
print(f"  max:    {merged['float_share_diff_pct'].max():.2f}%")

# If float-shares disagree systematically AND that explains mcap disagreement,
# we'd expect a tight relationship between the two diff_pct columns
print("\n=== Correlation between float-share diff and mcap diff ===")
print(merged[["float_share_diff_pct", "diff_pct"]].corr().iloc[0, 1])

=== Disagreement vs turnover ===
               count  median     max
turnover_rate                       
<0.1%              2    3.63    5.58
0.1-0.5%         224    2.08   51.06
0.5-1%           786    1.65  203.57
1-2%            1434    0.89  115.52
2-5%            1553    0.43  121.05
>5%              856    0.37  111.19

=== Top 15 worst disagreements ===
  ts_code  close_ts   close_bs    turn  turnover_rate  float_share  bs_implied_float_wan  float_share_diff_pct  circ_mv_yi  bs_circ_mv_yi   diff_pct
002594.SZ    282.66  93.110183  0.6677         0.6677  116245.7516         116249.453347              0.003184 3285.802415    1082.400783 203.566153
301016.SZ     20.96  14.598703  4.4239         2.8733    7176.0001           4660.828681             35.049768   15.040896       6.804205 121.052945
688603.SH    116.50  54.055417  1.6460         1.6460    2106.4583           2106.415553              0.002029   24.540239      11.386317 115.523938
301550.SZ     72.79  34.466720  5.4365 

In [3]:
import pandas as pd
cached = pd.read_csv("data/candidates/candidates_2022-01-17.csv")
fresh = pd.read_csv("data/candidates/candidates_2022-04-15.csv")
print(f"Cached: {len(cached.columns)} cols: {list(cached.columns)}")
print(f"Fresh:  {len(fresh.columns)} cols: {list(fresh.columns)}")

Cached: 9 cols: ['code', 'close', 'volume', 'amount', 'turn', 'tradestatus', 'isST', 'float_shares', 'float_mcap']
Fresh:  20 cols: ['ts_code', 'trade_date', 'turnover_rate', 'volume_ratio', 'pe', 'pe_ttm', 'pb', 'ps', 'total_share', 'float_share', 'total_mv', 'circ_mv', 'open', 'high', 'low', 'close', 'vol', 'amount', 'name', 'circ_mv_yi']


In [4]:
df = pd.read_csv("data/candidates/candidates_2024-12-16.csv")
df_sorted = df.sort_values("circ_mv_yi")
print("Smallest 10 by 流通市值 (亿 RMB):")
print(df_sorted[["ts_code", "name", "circ_mv_yi", "close", "turnover_rate"]].head(10).to_string(index=False))
print("\nLargest 5 (should be the megacaps like 工商银行, 茅台):")
print(df_sorted[["ts_code", "name", "circ_mv_yi"]].tail(5).to_string(index=False))

Smallest 10 by 流通市值 (亿 RMB):
  ts_code name  circ_mv_yi  close  turnover_rate
600898.SH  NaN    4.318157   1.71         2.0526
001226.SZ 拓山重工    4.926142  26.39         7.6077
600083.SH  NaN    5.175448   2.27         3.7493
301024.SZ 霍普股份    5.246753  29.58         8.5597
688638.SH 誉辰智能    5.284524  30.62         1.6021
301105.SZ 鸿铭股份    5.304380  32.27         4.8967
301113.SZ 雅艺科技    5.307904  28.74        13.9770
301234.SZ 五洲医疗    5.380016  29.22         5.5627
001259.SZ 利仁科技    5.414242  24.69         5.8079
301588.SZ 美新科技    5.505955  23.16         7.9079

Largest 5 (should be the megacaps like 工商银行, 茅台):
  ts_code name   circ_mv_yi
601988.SH 中国银行 11086.266081
601857.SH 中国石油 13682.415576
601288.SH 农业银行 16313.379171
601398.SH 工商银行 17605.677479
600519.SH 贵州茅台 19184.652802


In [5]:
import pandas as pd
sb = pd.read_csv("data/stock_basic.csv", dtype={"ts_code": str})
print("Is 600898 in stock_basic?", "600898.SH" in sb["ts_code"].values)
print("Is 600083 in stock_basic?", "600083.SH" in sb["ts_code"].values)

Is 600898 in stock_basic? False
Is 600083 in stock_basic? False
